In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os
import shutil
import time
import pandas as pd

# 1. Membangun SparkSession
spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

spark.version


26/11/08 20:46:56 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.1.194 instead (on interface wlp9s0)
26/11/08 20:46:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 20:46:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.9'

In [20]:
print("==Soal A==\n")
#Mendefinisikan skema
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

#Membuat folder simulasi streaming
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)#Menghapus sisa folder jika sudah ada sebelumnya
os.makedirs(folder_tugas, exist_ok=True)

#Mendefinisikan readStream
stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print(df_stream.isStreaming)

==Soal A==

True


In [21]:
print("==Soal B==\n")
#Menambahkan kolom turunan
turunan = stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

#Agregasi
agregasi_cabang = turunan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan_cabang"),
    count("pesanan_id").alias("jumlah_pesanan")
)

#Menentukan output sink dan menjalankan query streaming
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Status kueri:", query_cabang.status)

==Soal B==

Status kueri: {'message': 'Getting offsets from FileStreamSource[file:/home/cahyoalim/praktikum-bigdata/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


In [22]:
print("==Soal C==\n")
#Mendefinisikan ulang readStream untuk query kedua
stream_c = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

# membuat kolom turunan untuk kueri kedua
stream_c = stream_c.withColumn("total_harga", col("qty") * col("harga_satuan"))

#Agregasi berbasis window waktu
agregasi_waktu = stream_c.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan_waktu")
)

#Menentukan output sink dan menjalankan query
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Status kueri kedua :", query_waktu.status)

==Soal C==

Status kueri kedua : {'message': 'Getting offsets from FileStreamSource[file:/home/cahyoalim/praktikum-bigdata/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


In [23]:
print("==Soal D==\n")
import pandas as pd
import time
from datetime import datetime

# Mengambil waktu saat ini untuk kolom waktu_pesan
waktu_skrg = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Mengirim batch 1")
data_batch1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu Gula Aren", "Americano", "Latte"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 15000, 20000],
    "waktu_pesan": [waktu_skrg, waktu_skrg, waktu_skrg]
})
data_batch1.to_csv(f"{folder_tugas}/batch_1.csv", index=False)
time.sleep(7) 
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan_cabang DESC").show()


print("Mengirim batch 2")
data_batch2 = pd.DataFrame({
    "pesanan_id": ["P004", "P005"],
    "nama_cabang": ["Yogyakarta", "Semarang"],
    "nama_menu": ["Espresso", "V60 Arabica"],
    "qty": [1, 2],
    "harga_satuan": [12000, 22000],
    "waktu_pesan": [waktu_skrg, waktu_skrg]
})
data_batch2.to_csv(f"{folder_tugas}/batch_2.csv", index=False)
time.sleep(7)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan_cabang DESC").show()


print("Mengirim batch 3")
data_batch3 = pd.DataFrame({
    "pesanan_id": ["P006", "P007"],
    "nama_cabang": ["Solo", "Magelang"],
    "nama_menu": ["Caramel Macchiato", "Matcha Latte"],
    "qty": [2, 1],
    "harga_satuan": [25000, 24000],
    "waktu_pesan": [waktu_skrg, waktu_skrg]
})
data_batch3.to_csv(f"{folder_tugas}/batch_3.csv", index=False)
time.sleep(7)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan_cabang DESC").show()


print("Mengirim batch 4")
data_batch4 = pd.DataFrame({
    "pesanan_id": ["P008", "P009"],
    "nama_cabang": ["Semarang", "Purwokerto"],
    "nama_menu": ["Kopi Tubruk", "Kopi Susu Gula Aren"],
    "qty": [3, 2],
    "harga_satuan": [10000, 18000],
    "waktu_pesan": [waktu_skrg, waktu_skrg]
})
data_batch4.to_csv(f"{folder_tugas}/batch_4.csv", index=False)
time.sleep(7)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan_cabang DESC").show()


print("Mengirim batch 5")
data_batch5 = pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Yogyakarta", "Magelang"],
    "nama_menu": ["Red Velvet Latte", "Americano"],
    "qty": [1, 2],
    "harga_satuan": [22000, 15000],
    "waktu_pesan": [waktu_skrg, waktu_skrg]
})
data_batch5.to_csv(f"{folder_tugas}/batch_5.csv", index=False)
time.sleep(7)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan_cabang DESC").show()

spark.sql("SELECT window.start, window.end, * FROM agregasi_waktu").show(truncate=False)

==Soal D==

Mengirim batch 1
+-----------+-----------------------+--------------+
|nama_cabang|total_pendapatan_cabang|jumlah_pesanan|
+-----------+-----------------------+--------------+
|   Magelang|                  96000|             2|
|       Solo|                  15000|             1|
+-----------+-----------------------+--------------+

Mengirim batch 2
+-----------+-----------------------+--------------+
|nama_cabang|total_pendapatan_cabang|jumlah_pesanan|
+-----------+-----------------------+--------------+
|   Magelang|                  96000|             2|
|   Semarang|                  44000|             1|
|       Solo|                  15000|             1|
| Yogyakarta|                  12000|             1|
+-----------+-----------------------+--------------+

Mengirim batch 3
+-----------+-----------------------+--------------+
|nama_cabang|total_pendapatan_cabang|jumlah_pesanan|
+-----------+-----------------------+--------------+
|   Magelang|                 1200

In [24]:
print("Soal E\n")
import os
import pandas as pd
import time
from datetime import datetime

#Mendefinisikan dan menyalakan streaming query ketiga
query3= df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Status query3 :", query3.status)

#Mengirim batch data 
waktu_skrg = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("\n Mengirim batch 6")
data_batch_ekstra = pd.DataFrame({
    "pesanan_id": ["P999"],
    "nama_cabang": ["Magelang"],
    "nama_menu": ["Kopi Spesial Tugas"],
    "qty": [5],
    "harga_satuan": [50000],
    "waktu_pesan": [waktu_skrg]
})
data_batch_ekstra.to_csv(f"{folder_tugas}/batch_6_ekstra.csv", index=False)

time.sleep(10)

#Membuktikan berkas Parquet tersimpan
print("\n isi folder tugas10_output")
print(os.listdir("tugas10_output"))

#Membaca ulang berkas Parquet tersebut sebagai DataFrame batch
print("\nMembaca sebagai dataframe batch")
df_hasil_parquet = spark.read.parquet("tugas10_output")
df_hasil_parquet.show()

Soal E

Status query3 : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

 Mengirim batch 6

 isi folder tugas10_output
['.part-00000-f364e4a1-34b3-4893-86ab-5838036b52c3-c000.snappy.parquet.crc', 'part-00000-f364e4a1-34b3-4893-86ab-5838036b52c3-c000.snappy.parquet', '.part-00000-2ac63bc7-39de-4592-ab4b-2c684c4e7cfc-c000.snappy.parquet.crc', '.part-00002-dda23846-ab0f-447f-bfe8-0a8eaae50158-c000.snappy.parquet.crc', '.part-00003-87b8cf81-dcb7-4357-b1e4-f8e58f1bd208-c000.snappy.parquet.crc', '_spark_metadata', 'part-00001-4135c9ef-745f-42c2-8817-62d461962063-c000.snappy.parquet', '.part-00001-4135c9ef-745f-42c2-8817-62d461962063-c000.snappy.parquet.crc', 'part-00000-2ac63bc7-39de-4592-ab4b-2c684c4e7cfc-c000.snappy.parquet', '.part-00004-89db968f-9dc0-4d3b-9fc1-1571ad123fae-c000.snappy.parquet.crc', 'part-00002-dda23846-ab0f-447f-bfe8-0a8eaae50158-c000.snappy.parquet', 'part-00004-89db968f-9dc0-4d3b-9fc1-1571ad123fae-c000.snappy.parquet', 'part-0000

In [29]:
query_cabang.stop()

In [30]:
query_waktu.stop()

In [31]:
query3.stop()

In [32]:
spark.stop()